# Agents as state machines in LangGraph

An online electronics store handles return requests from its contact form: read the message, check the order,
check the warranty, then approve the refund or send it to a person. In this notebook you build that flow as a
real LangGraph graph, break it on purpose, fix it with a conditional edge, lose the log by dropping a reducer,
and make the warranty check fail once so a checkpointer can continue the run.

About 10 minutes. No API key is needed: the model's answers were recorded once and are replayed here.

## 1. Install LangGraph

LangGraph is not part of Colab. The next cell installs the versions the lesson ran (langgraph 1.2.21 with
langgraph-checkpoint 4.3.1).

**Look for:** no error. pip may print a note about other packages; you do not need to restart.

In [1]:
%pip install -q langgraph==1.2.21 langgraph-checkpoint==4.3.1

Note: you may need to restart the kernel to use updated packages.


## 2. The store: three messages, three orders, one warranty rule

All of this is the lesson's data. "Today" is fixed at 13 November 2026, so the warranty check gives the same
answer on every run. `find_order` is the order system: it returns the order, or `None` when there is no such
order.

**Look for:** the headphones are 7 months old, the laptop 25 months. And look closely at message c: the
customer wrote `EL-31590`, but the store's smart speaker order is `EL-31509`. Two digits are swapped.

In [2]:
import datetime as dt

TODAY = dt.date(2026, 11, 13)
WARRANTY_MONTHS = 12

ORDERS = {
    "EL-20417": {"item": "wireless headphones", "price": 129, "delivered": "2026-03-18"},
    "EL-18862": {"item": "14-inch laptop", "price": 899, "delivered": "2024-10-02"},
    "EL-31509": {"item": "smart speaker", "price": 59, "delivered": "2026-09-27"},
}

MESSAGES = {
    "a": "Hi, my wireless headphones from order EL-20417 stopped charging last week. "
         "I got them in March. Can I have my money back?",
    "b": "Hello. Order EL-18862, the laptop screen has started to flicker. "
         "I bought it a while ago but it was expensive and should not break like this. I want a refund.",
    "c": "Good morning, the smart speaker I ordered (order number EL-31590) will not turn on at all. "
         "Please refund me.",
}


def find_order(order_id):
    """The order system: the order, or None when there is no such order."""
    return ORDERS.get(order_id)


def months_since(day_iso):
    d = dt.date.fromisoformat(day_iso)
    return (TODAY.year - d.year) * 12 + (TODAY.month - d.month) - (1 if TODAY.day < d.day else 0)


for order_id, order in ORDERS.items():
    months = months_since(order["delivered"])
    print(f"{order_id}: {order['item']:<20} price {order['price']:>3}  {months} month{'' if months == 1 else 's'} since delivery")
for k, text in MESSAGES.items():
    print(f"message {k}: {text}")

EL-20417: wireless headphones  price 129  7 months since delivery
EL-18862: 14-inch laptop       price 899  25 months since delivery
EL-31509: smart speaker        price  59  1 month since delivery
message a: Hi, my wireless headphones from order EL-20417 stopped charging last week. I got them in March. Can I have my money back?
message b: Hello. Order EL-18862, the laptop screen has started to flicker. I bought it a while ago but it was expensive and should not break like this. I want a refund.
message c: Good morning, the smart speaker I ordered (order number EL-31590) will not turn on at all. Please refund me.


## 3. The model's recorded answers

In the lesson, a small hosted model (google/gemini-2.5-flash-lite, temperature 0, JSON output) read each
message and pulled out the order number "exactly as the customer wrote it". Its three answers are copied below
word for word. `recorded_reply` plays the role of the model call: it finds the answer for a message and parses
the JSON, the same way the lesson's script did.

**Look for:** for message c the model copied `EL-31590`, the customer's typo. The model did its job; it cannot
know which orders exist.

In [3]:
import json

# Recorded 2026-11-13 from openrouter, model google/gemini-2.5-flash-lite (the whole answer text, not altered)
RECORDED = {
    "a": "{\n  \"order_id\": \"EL-20417\",\n  \"problem\": \"Headphones not charging\"\n}",
    "b": "{\n\"order_id\": \"EL-18862\",\n\"problem\": \"flickering screen\"\n}",
    "c": "{\n  \"order_id\": \"EL-31590\",\n  \"problem\": \"smart speaker will not turn on\"\n}",
}


def recorded_reply(text):
    k = next(k for k, v in MESSAGES.items() if v == text)
    return json.loads(RECORDED[k])


for k in "abc":
    print(f"message {k} ->", recorded_reply(MESSAGES[k]))

message a -> {'order_id': 'EL-20417', 'problem': 'Headphones not charging'}
message b -> {'order_id': 'EL-18862', 'problem': 'flickering screen'}
message c -> {'order_id': 'EL-31590', 'problem': 'smart speaker will not turn on'}


## 4. The typed state and the five nodes

The state is a `TypedDict` you declare once. `log` uses `Annotated[list, operator.add]`: the function inside is
a **reducer**, it says how a new value joins the old one (`old + new` for lists). Each node is a plain function:
it gets the state and returns only the keys it changes. `MODEL_CALLS` counts how often `read_request` runs,
since each run would be a paid model call. `WARRANTY_SERVICE` lets you make the warranty check fail once later.

**Look for:** calling `check_order` by hand returns only two keys, `order` and `log`. For the typo order,
`order` is `None`.

In [4]:
import operator
from typing import Annotated, TypedDict


class Return(TypedDict):
    message: str
    order_id: str
    order: dict | None
    in_warranty: bool
    log: Annotated[list, operator.add]


MODEL_CALLS = []
WARRANTY_SERVICE = {"fail_next": False}


def read_request(state):
    MODEL_CALLS.append(state["message"][:20])
    reply = recorded_reply(state["message"])
    return {"order_id": reply["order_id"], "log": [f"model read order {reply['order_id']}"]}


def check_order(state):
    order = find_order(state["order_id"])
    return {"order": order, "log": [f"order {'found' if order else 'not found'}"]}


def check_warranty(state):
    if WARRANTY_SERVICE["fail_next"]:
        WARRANTY_SERVICE["fail_next"] = False
        raise TimeoutError("warranty service did not answer")
    months = months_since(state["order"]["delivered"])
    ok = months < WARRANTY_MONTHS
    return {"in_warranty": ok, "log": [f"{months} months since delivery"]}


def approve(state):
    return {"log": [f"refund {state['order']['price']} approved"]}


def escalate(state):
    return {"log": ["sent to a person"]}


print("check_order on EL-20417:", check_order({"order_id": "EL-20417"}))
print("check_order on EL-31590:", check_order({"order_id": "EL-31590"}))

check_order on EL-20417: {'order': {'item': 'wireless headphones', 'price': 129, 'delivered': '2026-03-18'}, 'log': ['order found']}
check_order on EL-31590: {'order': None, 'log': ['order not found']}


## 5. The graph builder

`build()` is the lesson's graph. `add_edge` is an arrow that is always taken. `add_conditional_edges` is a
fork: its routing function reads the state and names the next step. With `fixed=False` you get the graph drawn
straight from the policy text, with a single plain arrow out of `check_order`. `stream` prints what each node
returned, one line per step.

**Look for:** the arrows out of `check_order`. The policy graph has only `check_order -> check_warranty`; the
fixed graph also has `check_order -> escalate`.

In [5]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver


def after_order(state):
    return "found" if state["order"] else "not_found"


def after_warranty(state):
    return "approve" if state["in_warranty"] else "escalate"


def build(schema=Return, fixed=True, checkpointer=None):
    g = StateGraph(schema)
    g.add_node("read_request", read_request)
    g.add_node("check_order", check_order)
    g.add_node("check_warranty", check_warranty)
    g.add_node("approve", approve)
    g.add_node("escalate", escalate)
    g.add_edge(START, "read_request")
    g.add_edge("read_request", "check_order")
    if fixed:
        g.add_conditional_edges("check_order", after_order,
                                {"found": "check_warranty", "not_found": "escalate"})
    else:
        g.add_edge("check_order", "check_warranty")
    g.add_conditional_edges("check_warranty", after_warranty)
    g.add_edge("approve", END)
    g.add_edge("escalate", END)
    return g.compile(checkpointer=checkpointer)


def start(k):
    return {"message": MESSAGES[k], "log": []}


def stream(graph, inp, config=None):
    for update in graph.stream(inp, config, stream_mode="updates"):
        for node, change in update.items():
            print(f"  {node}: {json.dumps(change)}")


def arrows_out(graph, node):
    return sorted(f"{e.source} -> {e.target}" for e in graph.get_graph().edges if e.source == node)


print("policy graph:", arrows_out(build(fixed=False), "check_order"))
print("fixed graph: ", arrows_out(build(), "check_order"))

policy graph: ['check_order -> check_warranty']
fixed graph:  ['check_order -> check_warranty', 'check_order -> escalate']


## 6. Broken on purpose: the forgotten edge

Run message c through the graph drawn from the policy text. The cell catches the error and prints it, so the
notebook carries on.

**Look for:** two updates, then a `TypeError: 'NoneType' object is not subscriptable`, raised in
`check_warranty` on the line that reads `state["order"]["delivered"]`. LangGraph adds a note: "During task with
name 'check_warranty'". The error shows up in `check_warranty`, but the cause is earlier: `check_order` said
"not found" and had only one way out.

In [6]:
import traceback

broken = build(fixed=False)
try:
    stream(broken, start("c"))
except Exception as e:
    last = traceback.extract_tb(e.__traceback__)[-1]
    print("EXCEPTION:", type(e).__name__, e)
    print(f"raised in {last.name}: {last.line}")
    for note in getattr(e, "__notes__", []):
        print("note:", note)

  read_request: {"order_id": "EL-31590", "log": ["model read order EL-31590"]}
  check_order: {"order": null, "log": ["order not found"]}
EXCEPTION: TypeError 'NoneType' object is not subscriptable
raised in check_warranty: months = months_since(state["order"]["delivered"])
note: During task with name 'check_warranty' and id 'c3228d56-4459-ed68-6f40-3ab5ec2c27d0'


## 7. The fixed graph: a conditional edge after check_order

Now the three messages run through the fixed graph. For each one the cell streams the updates, then runs
`invoke` to get the final state.

**Look for:** message a goes to `approve` ("refund 129 approved"). Message b goes to `escalate` (25 months is
past the warranty). Message c prints 3 updates and skips `check_warranty`: its log ends with
"sent to a person". The final keys show that every node's partial update was merged into one state.

In [7]:
fixed = build()
for k in "abc":
    print(f"=== message {k}")
    stream(fixed, start(k))
    final = fixed.invoke(start(k))
    print("  final log:", json.dumps(final["log"]))
    print("  final keys:", sorted(final.keys()))

=== message a
  read_request: {"order_id": "EL-20417", "log": ["model read order EL-20417"]}
  check_order: {"order": {"item": "wireless headphones", "price": 129, "delivered": "2026-03-18"}, "log": ["order found"]}
  check_warranty: {"in_warranty": true, "log": ["7 months since delivery"]}
  approve: {"log": ["refund 129 approved"]}
  final log: ["model read order EL-20417", "order found", "7 months since delivery", "refund 129 approved"]
  final keys: ['in_warranty', 'log', 'message', 'order', 'order_id']
=== message b
  read_request: {"order_id": "EL-18862", "log": ["model read order EL-18862"]}
  check_order: {"order": {"item": "14-inch laptop", "price": 899, "delivered": "2024-10-02"}, "log": ["order found"]}
  check_warranty: {"in_warranty": false, "log": ["25 months since delivery"]}
  escalate: {"log": ["sent to a person"]}
  final log: ["model read order EL-18862", "order found", "25 months since delivery", "sent to a person"]
  final keys: ['in_warranty', 'log', 'message', 'o

## 8. Drop the reducer

The same graph, but `log` is a plain `list` with no reducer. Run message a again.

**Look for:** the final log holds only the last line, `["refund 129 approved"]`. No error and no warning:
without a reducer each update replaces the value, and the history is gone.

In [8]:
class ReturnNoReducer(TypedDict):
    message: str
    order_id: str
    order: dict | None
    in_warranty: bool
    log: list


nored = build(schema=ReturnNoReducer)
final = nored.invoke(start("a"))
print("final log:", json.dumps(final["log"]))

final log: ["refund 129 approved"]


## 9. A checkpointer, and a step that fails halfway

Compile the graph with `InMemorySaver`, a checkpointer that saves a snapshot of the state after each step,
under the thread id in `config`. Then run message b while the warranty service fails on its first call.

**Look for:** two updates, then `TimeoutError warranty service did not answer`. After that,
`get_state(config).next` is `('check_warranty',)`: the saved state knows which step comes next. The saved log
has the two finished steps, and the model was called once.

In [9]:
MODEL_CALLS.clear()
saved = build(checkpointer=InMemorySaver())
config = {"configurable": {"thread_id": "return-EL-18862"}}
WARRANTY_SERVICE["fail_next"] = True
try:
    stream(saved, start("b"), config)
except Exception as e:
    print("EXCEPTION:", type(e).__name__, e)
snap = saved.get_state(config)
print("state.next:", snap.next)
print("saved log:", json.dumps(snap.values["log"]))
print("model calls so far:", len(MODEL_CALLS))

  read_request: {"order_id": "EL-18862", "log": ["model read order EL-18862"]}
  check_order: {"order": {"item": "14-inch laptop", "price": 899, "delivered": "2024-10-02"}, "log": ["order found"]}
EXCEPTION: TimeoutError warranty service did not answer
state.next: ('check_warranty',)
saved log: ["model read order EL-18862", "order found"]
model calls so far: 1


## 10. Continue the same thread with None

The warranty service is back. Run the graph again with `None` as the input and the same `config`. `None` means
"no new input, continue this thread".

**Look for:** only two updates, `check_warranty` and `escalate`. `next` is now `()`: the run is finished. The
model was still called only once in all. The failed step itself ran again from its first line, so a step with a
side effect must be safe to run twice.

The second half of the cell is the same failure **without** a checkpointer: you can only run the whole request
again. Look for the same final log, but `model calls in all: 2`: every step before the failure ran again,
including the model call.

In [10]:
stream(saved, None, config)
snap = saved.get_state(config)
print("state.next:", snap.next)
print("final log:", json.dumps(snap.values["log"]))
print("model calls in all:", len(MODEL_CALLS))
print("checkpoints in history:", len(list(saved.get_state_history(config))))

print("=== the same failure, no checkpointer, the whole run again")
MODEL_CALLS.clear()
plain = build()
WARRANTY_SERVICE["fail_next"] = True
try:
    plain.invoke(start("b"))
except Exception as e:
    print("EXCEPTION:", type(e).__name__, e)
final = plain.invoke(start("b"))
print("final log:", json.dumps(final["log"]))
print("model calls in all:", len(MODEL_CALLS))

  check_warranty: {"in_warranty": false, "log": ["25 months since delivery"]}
  escalate: {"log": ["sent to a person"]}
state.next: ()
final log: ["model read order EL-18862", "order found", "25 months since delivery", "sent to a person"]
model calls in all: 1
checkpoints in history: 6
=== the same failure, no checkpointer, the whole run again
EXCEPTION: TimeoutError warranty service did not answer
final log: ["model read order EL-18862", "order found", "25 months since delivery", "sent to a person"]
model calls in all: 2


## 11. A checkpointer needs a thread id

A teammate calls `invoke` on the saved graph and forgets the `config`.

**Look for:** it fails at once with a `ValueError` that names `thread_id`. There is no default thread.

In [11]:
try:
    saved.invoke(start("a"))
except Exception as e:
    print("EXCEPTION:", type(e).__name__, str(e)[:400])

EXCEPTION: ValueError Checkpointer requires one or more of the following 'configurable' keys: thread_id, checkpoint_ns, checkpoint_id


## Now change it

Each cell below is ready to run. Edit the line marked `# <- edit`, run the cell again, and compare.

**Experiment 1: change the warranty rule.** The store now gives 30 months of warranty. Message b (the laptop,
25 months) should now take the other fork and end with "refund 899 approved", because `after_warranty` reads
`in_warranty`, which now comes out true. Put the line back to 12 and it goes to a person again. The cell resets
the rule to 12 at the end, so later cells are not affected.

In [12]:
WARRANTY_MONTHS = 30  # <- edit: try 12, 24, 26, 30
stream(build(), start("b"))
WARRANTY_MONTHS = 12  # back to the store's rule

  read_request: {"order_id": "EL-18862", "log": ["model read order EL-18862"]}
  check_order: {"order": {"item": "14-inch laptop", "price": 899, "delivered": "2024-10-02"}, "log": ["order found"]}
  check_warranty: {"in_warranty": true, "log": ["25 months since delivery"]}
  approve: {"log": ["refund 899 approved"]}


**Experiment 2: write your own reducer.** A reducer is any function of `(old, new)`. This one puts each new
line in front, so the log should come out in reverse order: "refund 129 approved" first. Try
`lambda old, new: old + new` to get `operator.add` back, or `lambda old, new: new` to see the no-reducer bug.

In [13]:
class ReturnNewestFirst(TypedDict):
    message: str
    order_id: str
    order: dict | None
    in_warranty: bool
    log: Annotated[list, lambda old, new: new + old]  # <- edit


final = build(schema=ReturnNewestFirst).invoke(start("a"))
print("final log:", json.dumps(final["log"]))

final log: ["refund 129 approved", "7 months since delivery", "order found", "model read order EL-20417"]


**Experiment 3: reuse a thread id.** Here message a runs on the thread `return-EL-18862`, the thread that
already holds request b. The checkpointer loads b's saved state first, and the reducer adds a's lines to it, so
the final log should hold b's four lines and then a's four. Change the thread id to `"return-EL-20417"` and the
log should have only a's four lines. One thread id per case.

In [14]:
thread_id = "return-EL-18862"  # <- edit: try "return-EL-20417"
cfg = {"configurable": {"thread_id": thread_id}}
final = saved.invoke(start("a"), cfg)
print("thread:", thread_id)
print("final log:", json.dumps(final["log"], indent=1))

thread: return-EL-18862
final log: [
 "model read order EL-18862",
 "order found",
 "25 months since delivery",
 "sent to a person",
 "model read order EL-20417",
 "order found",
 "7 months since delivery",
 "refund 129 approved"
]


## What you should see

- The graph with no "not found" arrow fails on request c with `TypeError: 'NoneType' object is not
  subscriptable`, during the task `check_warranty`.
- After adding the conditional edge, request c prints 3 updates and its log ends with `sent to a person`.
- With `log: list` and no reducer, request a's final log is `["refund 129 approved"]`.
- After the timeout, `get_state(config).next` is `('check_warranty',)`, and running the graph again with `None`
  as input runs only `check_warranty` and `escalate`.
- Invoking the saved graph without a config raises a `ValueError` that names `thread_id`.